# import packages

In [167]:
#pip install pdfminer.high_level
#pip install keybert

In [168]:
from pdfminer.high_level import extract_text
import csv
import pandas as pd
from keybert import KeyBERT
import yake
import spacy
import pytextrank

# read in CV

In [169]:
# Path to the PDF file
pdf_file_path = 'Ayush_Shrestha_CV.pdf'

In [170]:
# Extract text from the PDF
text = extract_text(pdf_file_path)

# Create a DataFrame with the path and text
df = pd.DataFrame({
    'file_path': [pdf_file_path],
    'content': [text]
})

In [171]:
#check
df

,file_path,content
0,Ayush_Shrestha_CV.pdf,AYUSH SHRESTHA\n\nS t u d e n t\n\nABOUT ME\n\...


# get keywords out of cv

In [172]:
# Get the document text
doc_text = df['content'][0]

In [173]:
#try 1

In [174]:
# KeyBERT
kw_model = KeyBERT()
keywords_keybert = kw_model.extract_keywords(doc_text)
keywords_keybert = set([kw[0] for kw in keywords_keybert])
keywords_keybert

{'academic', 'assistant', 'data', 'datasets', 'programming'}

In [175]:
#try 2

In [176]:
# PyTextRank
nlp = spacy.load("en_core_web_sm")
nlp.add_pipe("textrank")
doc = nlp(doc_text)
keywords_pytextrank = set([phrase.text for phrase in doc._.phrases[:10]])
keywords_pytextrank

{'Big Data',
 'Business Engineering',
 'Everest Analytics',
 'box office earnings',
 'hotel room price prediction kaggle winner',
 'hotel room prices',
 'social media pages',
 'social media sentiments',
 'valuable business insights',
 'various machine'}

In [177]:
#try 3

In [178]:
# YAKE
kw_extractor = yake.KeywordExtractor()
keywords_yake = kw_extractor.extract_keywords(doc_text)
keywords_yake = set([kw[0] for kw in keywords_yake])
keywords_yake

{'AYUSH',
 'AYUSH SHRESTHA',
 'Analytics',
 'Business Engineering',
 'Data',
 'Data Science',
 'Data Science program',
 'MSc Business Engineering',
 'Private Tutor',
 'Rijbewijs B EDUCATION',
 'SHRESTHA',
 'Science',
 'Science program',
 'future graduate',
 'machine learning',
 'media',
 'passion lies',
 'social',
 'social media',
 'social media data'}

In [179]:
# Combine all keywords into a single document for re-extraction
combined_keywords_text = keywords_keybert.union(keywords_pytextrank).union(keywords_yake)
combined_keywords_text = str(combined_keywords_text)

# Re-extract keywords from the combined text using KeyBERT
combined_keywords_bert = kw_model.extract_keywords(combined_keywords_text)
overlapping_keywords_bert = [kw[0] for kw in combined_keywords_bert]

# Re-extract keywords from the combined text using YAKE
combined_keywords_yake = kw_extractor.extract_keywords(combined_keywords_text)
overlapping_keywords_yake = [kw[0] for kw in combined_keywords_yake]

# Re-extract keywords from the combined text using Pagerank
combined_keywords_pagerank = nlp(combined_keywords_text)
overlapping_keywords_pagerank = set([phrase.text for phrase in doc._.phrases[:10]])

# Output the results
print("KeyBERT Keywords:", keywords_keybert)
print("PyTextRank Keywords:", keywords_pytextrank)
print("YAKE Keywords:", keywords_yake)
#print("Overlapping Keywords from KeyBERT re-extraction:", overlapping_keywords_bert)
print("Overlapping Keywords from YAKE re-extraction:", overlapping_keywords_yake) #as yake seems to give the best results
#print("Overlapping Keywords from pagerank re-extraction:", overlapping_keywords_pagerank)

KeyBERT Keywords: {'academic', 'assistant', 'data', 'programming', 'datasets'}
PyTextRank Keywords: {'hotel room prices', 'box office earnings', 'various machine', 'Big Data', 'hotel room price prediction kaggle winner', 'valuable business insights', 'Everest Analytics', 'social media pages', 'Business Engineering', 'social media sentiments'}
YAKE Keywords: {'social', 'MSc Business Engineering', 'Science program', 'Data Science program', 'future graduate', 'passion lies', 'Rijbewijs B EDUCATION', 'Data', 'Analytics', 'AYUSH', 'SHRESTHA', 'AYUSH SHRESTHA', 'Science', 'Private Tutor', 'Data Science', 'media', 'social media data', 'machine learning', 'Business Engineering', 'social media'}
Overlapping Keywords from YAKE re-extraction: ['MSc Business Engineering', 'Data Science program', 'social media data', 'social media sentiments', 'social media pages', 'valuable business insights', 'hotel room price', 'box office earnings', 'prediction kaggle winner', 'room price prediction', 'price pr

In [180]:
df['keyword'] = [overlapping_keywords_yake]

# save data 

In [181]:
df

,file_path,content,keyword
0,Ayush_Shrestha_CV.pdf,AYUSH SHRESTHA\n\nS t u d e n t\n\nABOUT ME\n\...,"[MSc Business Engineering, Data Science progra..."


In [182]:
csv_file_path = 'extracted_cvs.csv'

# If the CSV file does not exist, create it and include the header
# If it exists, append without the header
header_needed = not pd.io.common.file_exists(csv_file_path)

# Save or append to CSV
df.to_csv(csv_file_path, mode='a', index=False, header=header_needed)